# Step 2 — Extract and inspect the 53-entry pilot

Use the **same 53 personal entries on printed page 161**, from Aalto J through Abberton T Frank. Run step 1 first. This notebook retrieves existing Internet Archive word OCR and runs a small Python rule-based baseline; it does not call a new OCR/model API.

**Purpose:** produce a traceable first-pass table and concrete examples for the client meeting. Missing OCR and incorrect values remain visible. The 15 assistant image checks are development references, pending human review.


In [1]:
from pathlib import Path
import csv
import html
import subprocess
import sys
from IPython.display import HTML, Markdown, display

candidates = [
    base / prefix / "fa26/scripts/extract_a_sample.py"
    for base in (Path.cwd(), *Path.cwd().parents)
    for prefix in ("", "ds-bpl-city-directories")
]
script = next((path for path in candidates if path.is_file()), None)
if script is None:
    raise FileNotFoundError("Open this notebook inside ds-bpl-city-directories.")
output = script.parents[1] / "data/a_sample"
if not (output / "sample_index.csv").exists():
    raise FileNotFoundError("Run 01_download_a_sample.ipynb first.")
print("Script:", script)
print("Sample folder:", output)


Script: /Users/yly/Documents/ChatGPT/Spark BPL Project/ds-bpl-city-directories/fa26/scripts/extract_a_sample.py
Sample folder: /Users/yly/Documents/ChatGPT/Spark BPL Project/ds-bpl-city-directories/fa26/data/a_sample


## Run the baseline

Reruns reuse cached OCR and replace the same five files in `extraction/`. Keep persistent review edits in `review/image_checks.json`. They are compared with the baseline and never copied into its extracted fields.


In [2]:
subprocess.run([sys.executable, str(script)], check=True)


Cached: p161_ocr.json
Matched 51/53 selected records to OCR candidates.
Image-checked 15 records; 34 records in the review queue.
Results: /Users/yly/Documents/ChatGPT/Spark BPL Project/ds-bpl-city-directories/fa26/data/a_sample/extraction


CompletedProcess(args=['/Users/yly/Desktop/DS701/.venv/bin/python', '/Users/yly/Documents/ChatGPT/Spark BPL Project/ds-bpl-city-directories/fa26/scripts/extract_a_sample.py'], returncode=0)

## Inspect the structured records

`selection_label` is only a navigation aid. `surname` and `given_name` are the names actually recovered from OCR/context. A matched candidate can still contain errors. Blank fields are not evidence that the original directory lacked that information.


In [3]:
def read_csv(path):
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def show_table(rows, fields):
    header = "".join(f"<th>{html.escape(field)}</th>" for field in fields)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(row[field])}</td>" for field in fields) + "</tr>"
        for row in rows
    )
    display(HTML(f'<div style="overflow-x:auto"><table><thead><tr>{header}</tr></thead>'
                 f'<tbody>{body}</tbody></table></div>'))

records = read_csv(output / "extraction/records.csv")
checks = read_csv(output / "extraction/check_results.csv")
issues = read_csv(output / "extraction/issues.csv")
print("Selected entries:", len(records))
print("Matched OCR candidates:", sum(row["extraction_status"] == "candidate" for row in records))
print("Review queue:", len(issues), "(flags are not necessarily errors)")
show_table(records[:8], ["sample_id", "surname", "given_name", "occupation_raw",
                        "work_address_raw", "residence_address_raw", "image_check_status"])


Selected entries: 53
Matched OCR candidates: 51
Review queue: 34 (flags are not necessarily errors)


sample_id,surname,given_name,occupation_raw,work_address_raw,residence_address_raw,image_check_status
A001,Aalto,J,tailor,,61 Myrtle,assistant_checked_agreement
A002,Aaron,Abraham L,jeweler,1915 Wash,624 Warren Rox,assistant_checked_agreement
A003,Aaron,Barney,com trav,,668 Blue Hill av Dor,assistant_checked_mismatch
A004,Aaron,Charles T,salesman,,18 Haviland,not_checked
A005,Aaron,David J,lawyer,40 Court rm 1002,1254 Conunonwealth av,not_checked
A006,Aaron,Fred,cigars,,77 Empire Alls,not_checked
A007,Aaron,Harry F,salesman,40 Central rm 305,737 E Broadway,assistant_checked_agreement
A008,Aaron,Herman,mgr,628 E Fourth S B,737 E Broadway do,assistant_checked_agreement


## Follow examples back to the scan

A001 illustrates a simple entry; A002 illustrates the raised continuation; A003 illustrates an OCR number error; A031 illustrates a missing entry. `raw_ocr_text` keeps the candidate's physical OCR line group, which can include a raised fragment belonging to the next entry. `working_text` contains the proposed reconstructed entry. `source_line_ids` includes donor lines as well as the entry's own lines.


In [4]:
for sample_id in ("A001", "A002", "A003", "A031"):
    row = next(record for record in records if record["sample_id"] == sample_id)
    display(Markdown(f"### {sample_id}: {row['selection_label']}"))
    print("Original scan:", output / row["local_image"])
    print("Source viewer:", row["source_page_url"])
    print("OCR lines:", row["source_line_ids"] or "Not recovered")
    print("Physical OCR:", row["raw_ocr_text"] or "Not recovered")
    print("Working text:", row["working_text"] or "Not recovered")
    show_table([row], ["surname", "given_name", "occupation_normalized",
                       "residence_address_normalized", "review_flags"])


### A001: Aalto J

Original scan: /Users/yly/Documents/ChatGPT/Spark BPL Project/ds-bpl-city-directories/fa26/data/a_sample/images/bostonmassachuse1916112samp_leaf0163_p161.jpg
Source viewer: https://archive.org/details/bostonmassachuse1916112samp/page/161/mode/1up
OCR lines: L013
Physical OCR: Aalto J tailor h 61 Myrtle [ren Rox
Working text: Aalto J tailor h 61 Myrtle


surname,given_name,occupation_normalized,residence_address_normalized,review_flags
Aalto,J,tailor,61 Myrtle,bracket_continuation_rule


### A002: Aaron Abraham L

Original scan: /Users/yly/Documents/ChatGPT/Spark BPL Project/ds-bpl-city-directories/fa26/data/a_sample/images/bostonmassachuse1916112samp_leaf0163_p161.jpg
Source viewer: https://archive.org/details/bostonmassachuse1916112samp/page/161/mode/1up
OCR lines: L013;L014
Physical OCR: Aaron Abraham L jeweler 1915 Wash h 624 War-
Working text: Aaron Abraham L jeweler 1915 Wash h 624 Warren Rox


surname,given_name,occupation_normalized,residence_address_normalized,review_flags
Aaron,Abraham L,jeweler,"624 Warren, Roxbury",bracket_continuation_rule


... (2 outputs left out)

surname,given_name,occupation_normalized,residence_address_normalized,review_flags
Aaron,Barney,commercial traveler,"668 Blue Hill Avenue, Dorchester",


### A031: Abate Angelo

Original scan: /Users/yly/Documents/ChatGPT/Spark BPL Project/ds-bpl-city-directories/fa26/data/a_sample/images/bostonmassachuse1916112samp_leaf0163_p161.jpg
Source viewer: https://archive.org/details/bostonmassachuse1916112samp/page/161/mode/1up
OCR lines: Not recovered
Physical OCR: Not recovered
Working text: Not recovered


surname,given_name,occupation_normalized,residence_address_normalized,review_flags
,,,,selected_entry_not_found_in_ocr


## Review the omissions and image disagreements

The values below come from assistant inspection of the scan. A teammate should verify them before they become human reference answers. These checks deliberately include difficult cases and were used during development: **they are not a held-out accuracy estimate**.


In [5]:
missing = [row for row in records if row["extraction_status"] == "not_recovered"]
print("Selected entries not recovered from existing OCR:")
show_table(missing, ["sample_id", "selection_label", "extraction_status", "source_page_url"])

disagreements = [row for row in checks if row["agreement"] == "no"]
print("Assistant image-check disagreements:")
show_table(disagreements, ["sample_id", "field", "baseline_value", "expected_from_image"])


Selected entries not recovered from existing OCR:


sample_id,selection_label,extraction_status,source_page_url
A031,Abate Angelo,not_recovered,https://archive.org/details/bostonmassachuse1916112samp/page/161/mode/1up
A049,Abbe John W,not_recovered,https://archive.org/details/bostonmassachuse1916112samp/page/161/mode/1up


Assistant image-check disagreements:


sample_id,field,baseline_value,expected_from_image
A003,residence_address_raw,668 Blue Hill av Dor,568 Blue Hill av Dor
A003,residence_address_normalized,"668 Blue Hill Avenue, Dorchester","568 Blue Hill Avenue, Dorchester"
A020,surname,,Aaronson
A020,residence_address_raw,,177 Magnolia Dor
A020,residence_address_normalized,,"177 Magnolia, Dorchester"
A020,neighborhood_raw,,Dor
A020,neighborhood_normalized,,Dorchester
A021,surname,,Aaronson
A021,status_detail_raw,Aug Aug 31 3 1915,Aug 31 1915
A031,surname,,Abate


## Meeting summary

The summary is regenerated by the script. The field definitions, normalization rules, limitations and source numbering are documented in `fa26/data/a_sample/README.md`.


In [6]:
display(Markdown((output / "extraction/report.md").read_text(encoding="utf-8")))


# Page 161 extraction pilot

- Selected personal entries: **53**.
- Entries matched to existing OCR candidates: **51**.
- Selected entries not recovered: **2**.
- Records with review flags / observed disagreements: **34**.
- Assistant image checks: **15** records; **9** agree on all checked fields.
- Team/human-verified gold-standard records: **0** in this workflow.

## What was run

Existing Internet Archive word OCR → spatial line ordering → indentation-based entry grouping → bracket-continuation handling → surname context and regex field parsing → selected-record matching.

The known 53-name list is used only to select/match OCR candidates. Its names and status notes are never substituted for missing extracted fields. This is a sample-assisted development baseline, not a blind evaluation of finding every person on an unseen page.

Raw OCR JSON remains unchanged. `raw_ocr_text` preserves the reconstructed physical lines, including OCR mistakes. `working_text` joins wrapping and moves candidate raised-bracket continuations to the following entry. Both versions are retained; `source_line_ids` also includes any preceding physical line supplying a raised continuation. No paid model API or geocoder was called.

The current layout settings, abbreviation rules and surname handling are specific to this A-section pilot. Field confidence scores and geocoding have not been implemented.

## Image-check results

These are assistant-produced development spot checks selected to include difficult cases. They are not independent human gold labels, a held-out test, or population accuracy. The parser does not read expected values until its output is complete. Exact agreement ignores whitespace only; correctness of a blank field is checked explicitly when included.

| Field | Agreement / checked |
|---|---:|
| surname | 12 / 15 |
| given_name | 14 / 15 |
| name_qualifier | 15 / 15 |
| occupation_raw | 14 / 15 |
| occupation_normalized | 14 / 15 |
| affiliation_raw | 15 / 15 |
| work_address_raw | 15 / 15 |
| work_address_normalized | 15 / 15 |
| residence_type_raw | 14 / 15 |
| residence_type_normalized | 14 / 15 |
| residence_address_raw | 11 / 15 |
| residence_address_normalized | 11 / 15 |
| neighborhood_raw | 12 / 15 |
| neighborhood_normalized | 12 / 15 |
| record_status | 14 / 15 |
| status_detail_raw | 13 / 15 |

## Main issues to discuss

- Missing surname anchors can propagate the wrong surname through ditto entries.
- Entire entries and address continuations are missing from the existing OCR.
- Raised-bracket text may physically sit beside the preceding record.
- `do` can repeat a full work address or only the work-address neighborhood.
- Death/removal notices remain records; they are not counted as confirmed current residents.
- Commodity/activity words such as `cigars` do not establish a specific occupation.

## Decisions for the client

1. Confirm the representation and analytical treatment of death/removal notices.
2. Confirm how to retain activity-only occupation descriptions without inventing a role.
3. Review the raw/normalized schema and treatment of locality-only residences.

## Reproduce and inspect

Run `python3 fa26/scripts/extract_a_sample.py` or the step-2 notebook. See `../README.md` for fields and limitations. `records.csv` is generated; review notes live in `../review/image_checks.json`, outside the generated output folder.

OCR input: https://iiif.archive.org/iiif/3/annotations/bostonmassachuse1916112samp/bostonmassachuse1916112samp_djvu.xml/163.json
OCR SHA-256: `6cdd032291dbd76c11c7430fd2a7c918e6f6ef079c628db41420d81269a68edd`


## What the team does next

1. Verify the 15 image references and resolve the concrete differences against the original scan. Keep reviewer attribution with those checks.
2. Bring a few scan → OCR → fields examples and the short report to the client meeting. Confirm the schema and how to treat notices/activity-only occupation descriptions/locality-only residences.
3. After the conventions are agreed, expand the reviewed reference set and decide whether another OCR pass is needed before scaling up.

**Current boundary:** this is a page-specific extraction pilot, not a final analysis-ready dataset. There are no coordinates, calibrated confidence scores or inferred current addresses. A separate batch of reference records from another page cannot be used as answer labels for these 53 people.
